# 02 - Entrenamiento TabNet

Este notebook carga desde `data/optuna` los mejores hiperparámetros, entrena los modelos de clasificación y regresión durante 200 épocas y guarda únicamente ambos modelos.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Daprosero/chec_impacto.git"
REPO_NAME = "chec_impacto"

def resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "chec_impacto").exists():
            return candidate

    working_root = Path("/kaggle/working") if Path("/kaggle/working").exists() else cwd
    clone_dir = working_root / REPO_NAME
    if not clone_dir.exists():
        subprocess.run(
            ["git", "clone", REPO_URL, str(clone_dir)],
            check=True,
        )
    return clone_dir.resolve()


def install_project_requirements(project_root):
    requirements_path = project_root / "requirements.txt"
    if not requirements_path.exists():
        raise FileNotFoundError(f"No existe requirements.txt en {project_root}")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)],
        check=True,
    )


def ensure_lfs_data(project_root):
    data_path = project_root / "data" / "Indicadores_vano_v3.csv"
    if shutil.which("git-lfs") and (project_root / ".git").exists():
        subprocess.run(["git", "lfs", "install", "--local"], cwd=project_root, check=False)
        subprocess.run(["git", "lfs", "pull"], cwd=project_root, check=True)

    if data_path.exists() and data_path.stat().st_size < 1024:
        head = data_path.read_text(errors="ignore")[:120]
        if "git-lfs" in head:
            raise RuntimeError(
                "Indicadores_vano_v3.csv quedó como puntero Git LFS. "
                "Activa Git LFS en el entorno o descarga el archivo LFS antes de continuar."
            )


PROJECT_ROOT = resolve_project_root()
install_project_requirements(PROJECT_ROOT)
ensure_lfs_data(PROJECT_ROOT)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

os.chdir(PROJECT_ROOT)
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

import torch

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("CUDA disponible:", torch.cuda.is_available())
print("MPS disponible:", torch.backends.mps.is_available())

In [ ]:
from chec_impacto.data import preparar_splits_estratificados, procesar_dataset_completo
from chec_impacto.models import resolve_tabnet_device

VENTANA_CLIMATICA_HORAS = 12
FILTRO_UITI_MAX = 1000

DATA_PATH_CANDIDATES = [
    DATA_DIR / "Indicadores_vano_v3.csv",
    DATA_DIR / "Indicadores_vano_v2.csv",
    DATA_DIR / "Indicadores_vano_v1.csv",
]
DATASET_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)
if DATASET_PATH is None:
    raise FileNotFoundError("No se encontro Indicadores_vano_v3/v2/v1.csv en data/.")

VARIABLES_SELECCION_PATH = DATA_DIR / "Variables_seleccion.xlsx"
DEVICE_NAME = resolve_tabnet_device()
print(f"Usando device: {DEVICE_NAME}")

datos_procesados = procesar_dataset_completo(
    path_clima=DATASET_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target="UITI_VANO",
    filtro_uiti_max=FILTRO_UITI_MAX,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

X = datos_procesados["X"]
y = datos_procesados["y"]
features = datos_procesados["features"]
categorical_dims = datos_procesados["categorical_dims"]
df = datos_procesados["df_final"]
CATEGORICAL_COLUMNS = datos_procesados["CATEGORICAL_COLUMNS"]

splits_por_modo = {
    modo: preparar_splits_estratificados(X, y, modo=modo)
    for modo in ("clasificacion", "regresion")
}


In [ ]:
from chec_impacto.training import (
    cargar_estudio_optuna,
    cargar_o_entrenar_tabnet,
    get_model_paths,
)

OPTUNA_DIR = DATA_DIR / "optuna"
KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_DIR = (
    KAGGLE_WORKING / "models"
    if KAGGLE_WORKING.exists()
    else DATA_DIR / "models"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OPTUNA_STUDY_FILES = {
    "clasificacion": "tabnet_optuna_study_clasificacion.pkl",
    "regresion": "tabnet_optuna_study_regresion.pkl",
}
OPTUNA_JOURNAL_FILES = {
    "clasificacion": "tabnet_classification_params.journal",
    "regresion": "tabnet_regression_params.journal",
}
MAX_EPOCHS = 200
PATIENCE = 200

study_paths = {
    modo: OPTUNA_DIR / filename
    for modo, filename in OPTUNA_STUDY_FILES.items()
    if (OPTUNA_DIR / filename).exists()
}
if not study_paths:
    raise FileNotFoundError(
        "No se encontro ningun estudio Optuna en data/optuna."
    )
print("Modos disponibles:", list(study_paths))

# Los Study existentes fueron serializados cuando los journals estaban en data/.
# Se conserva esa ruta para deserializarlos; las búsquedas nuevas usan data/optuna/.
for modo in study_paths:
    journal_name = OPTUNA_JOURNAL_FILES[modo]
    source = OPTUNA_DIR / journal_name
    legacy_path = DATA_DIR / journal_name
    if not source.exists():
        raise FileNotFoundError(f"Falta el journal Optuna requerido: {source}")
    if not legacy_path.exists():
        shutil.copy2(source, legacy_path)

for modo in study_paths:
    study_path = study_paths[modo]
    print(f"\nCargando Study de {modo}: {study_path}")
    study = cargar_estudio_optuna(study_path)
    if not study.trials:
        raise ValueError(f"El Study de {modo} no contiene trials: {study_path}")

    splits = splits_por_modo[modo]
    paths = get_model_paths(modo, OUTPUT_DIR)
    cargar_o_entrenar_tabnet(
        modo_objetivo=modo,
        params=study.best_params,
        paths=paths,
        X_train=splits["X_train"],
        y_train=splits["y_train"],
        X_valid=splits["X_valid"],
        y_valid=splits["y_valid"],
        features=features,
        categorical_columns=CATEGORICAL_COLUMNS,
        categorical_dims=categorical_dims,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
    )

model_files = sorted(OUTPUT_DIR.glob("*.zip"))
if len(model_files) != len(study_paths):
    raise RuntimeError(f"Se esperaban {len(study_paths)} modelos .zip y se encontraron: {model_files}")

print(f"\nModelos guardados en: {OUTPUT_DIR}")
for model_path in model_files:
    print(model_path.name)
